# 45. Quantization, KV cache, and inference optimization

**Level:** Advanced

**Prerequisites:** 35-39: transformer attention and decoding.

[Course index](00_START_HERE.ipynb) | [Previous](44_FineTuning_LoRA_Distillation_and_Preferences.ipynb) | [Next](46_Generative_Models_Autoencoders_and_Diffusion.ipynb)

**Topics:** Quantization, Inference optimization, KV cache, Batching, Prefill, Decode, Memory.

**Matching script:** `examples\45_Quantization_KV_Cache_and_Inference_Cost.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

Quantization represents values with fewer bits using a scale and possibly a zero point.
Lower storage precision can reduce weight memory, but accuracy, latency, supported kernels,
and device behavior depend on the method. Compressing a NumPy array is not proof of faster
model inference. Distinguish weight-only and activation quantization.

Autoregressive inference has a prefill phase processing the prompt and a decode phase adding
tokens. A KV cache reuses attention keys and values from earlier positions rather than
recomputing every projection. It consumes memory proportional to sequence length, batch size,
layers, KV heads, head dimension, and value precision.

Batching improves throughput but can increase waiting latency; caching repeated prompts can
reduce work subject to model/service policy. Attention implementations, grouped-query heads,
paging, speculative decoding, and hardware kernels affect costs. Speculative decoding must
preserve the target distribution under its acceptance algorithm, not merely accept drafts blindly.

## Mathematics or data contract

Approximate KV bytes = 2 * layers * batch * tokens * kv_heads * head_dim * bytes_per_value.

## Important limitation

The symmetric int8 exercise checks storage/error arithmetic, not production quantization speed or perplexity.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Quantize a weight array and estimate cache memory

**Purpose:** Separate numerical error, payload bytes, and runtime claims.

### Read the code in this order
1. Choose a symmetric scale from maximum absolute weight.
2. Round and clip to int8.
3. Dequantize and bound reconstruction error.
4. Compute a transparent KV memory estimate with explicit units.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.
- A fixed random seed makes the chosen sampling reproducible within the environment.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
float/int8 payload bytes: 4096 1024 (scale metadata additional)
maximum weight error: 0.00287
example KV estimate MiB: 24.0
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import numpy as np

weights = np.random.default_rng(42).normal(0, 0.2, (32, 32)).astype(np.float32)
scale = float(np.abs(weights).max()) / 127
quantized = np.clip(np.rint(weights / scale), -127, 127).astype(np.int8)
restored = quantized.astype(np.float32) * scale
error = float(np.max(np.abs(weights - restored)))
assert error <= scale / 2 + 1e-7
assert quantized.nbytes == weights.nbytes // 4
layers, batch, tokens, kv_heads, head_dim, bytes_per_value = 12, 1, 2048, 4, 64, 2
cache_bytes = 2 * layers * batch * tokens * kv_heads * head_dim * bytes_per_value
assert cache_bytes > 0
print("float/int8 payload bytes:", weights.nbytes, quantized.nbytes, "(scale metadata additional)")
print("maximum weight error:", round(error, 6))
print("example KV estimate MiB:", round(cache_bytes / 1024**2, 2))

float/int8 payload bytes: 4096 1024 (scale metadata additional)
maximum weight error: 0.00287
example KV estimate MiB: 24.0


### Interpret example 1

The result should demonstrate this purpose: **Separate numerical error, payload bytes, and runtime claims.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Compare int8 error across an outlier-heavy weight distribution and a per-channel scaling variant.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Quantization` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: Approximate KV bytes = 2 * layers * batch * tokens * kv_heads * head_dim * bytes_per_value.
- Name the boundary of the demonstration: The symmetric int8 exercise checks storage/error arithmetic, not production quantization speed or perplexity.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://arxiv.org/abs/2305.14314`
- `https://arxiv.org/abs/2309.06180`
- `https://docs.pytorch.org/tutorials/beginner/basics/intro.html`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.